# Cross-Market Volatility Forecasting

دفتر Colab لتشغيل مقارنة توقع التباين اليومي لسوقي S&P 500 وTASI باستخدام اختبارات زمنية متتابعة ونماذج إحصائية وتعلم آلي. المتغير المستهدف هو مربع العائد اللوغاريتمي للجلسة التالية، وهو بديل ضوضائي للتقلب الكامن؛ النتائج ليست توصية استثمارية.

البيانات الخام غير مرفقة بالمستودع. استخدم ملفات أسعار مرخّصة لديك، وتحقق من شروط مزود البيانات.

## 1. استنساخ المستودع وتثبيت المتطلبات

بعد نشر التحديثات على GitHub، شغّل الخلايا التالية.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = 'https://github.com/alrashedinaifrr/financial-volatility-forecasting.git'
PROJECT_DIR = Path('/content/financial-volatility-forecasting')
if not (PROJECT_DIR / 'src' / 'portfolio_v7.py').exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_DIR)], check=True)
assert (PROJECT_DIR / 'src' / 'portfolio_v7.py').exists()
print('Project:', PROJECT_DIR)

In [ ]:
%cd /content/financial-volatility-forecasting
%pip install -q -r requirements.txt

## 2. إضافة ملفات الأسعار

ارفع أو انسخ ملفي `sp500.csv` و`tasi.csv` إلى مجلد Drive. يجب أن يحتوي كل ملف على عمودي `Date` و`Close`، وأن تكون الأسعار مرتبة أو قابلة للترتيب زمنيًا. لا ترفع بيانات سرية.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = Path('/content/drive/MyDrive/volatility_data')
for filename in ['sp500.csv', 'tasi.csv']:
    path = DATA_DIR / filename
    print(('✓' if path.is_file() else '✗'), path)
if not all((DATA_DIR / f).is_file() for f in ['sp500.csv','tasi.csv']):
    raise FileNotFoundError('ضع sp500.csv وtasi.csv في المجلد المحدد أو عدّل DATA_DIR.')

In [ ]:
import pandas as pd
for filename in ['sp500.csv','tasi.csv']:
    df = pd.read_csv(DATA_DIR / filename)
    missing = {'Date','Close'} - set(df.columns)
    if missing: raise ValueError(f'{filename} missing columns: {sorted(missing)}')
    print(filename, 'rows=', len(df), 'date range=', df.Date.min(), '→', df.Date.max())
    display(df[['Date','Close']].head())

## 3. تشغيل الاختبارات والتحليل

قد يستغرق التحليل بعض الوقت بسبب إعادة أخذ العينات bootstrap. يسجل التشغيل بصمة SHA-256 للمدخلات وإصدارات الحزم.

In [ ]:
subprocess.run(['python', '-m', 'pytest', '-q', 'tests'], cwd=PROJECT_DIR, check=True)
OUTPUT_DIR = Path('/content/volatility_results')
subprocess.run([
    'python', str(PROJECT_DIR / 'src' / 'portfolio_v7.py'),
    '--data-dir', str(DATA_DIR),
    '--output-dir', str(OUTPUT_DIR),
], check=True)

In [ ]:
from IPython.display import display, Image
for name in ['fold_scores.csv','pairwise_qlike_comparisons.csv','aligned_transfer.csv','run_metadata.json']:
    path = OUTPUT_DIR / name
    print('\n', name)
    if path.suffix == '.csv': display(pd.read_csv(path).head(20))
    else: print(path.read_text()[:4000])
for path in sorted(OUTPUT_DIR.glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/volatility_results', 'zip', OUTPUT_DIR)
print('Results archive:', archive)
files.download(archive)

## حدود التفسير

- اختلاف التقاويم والبنية السوقية يمنع اعتبار أرقام الأسواق مقارنة مباشرة لقابلية التنبؤ.
- فواصل bootstrap وصفية واستكشافية؛ تصحيح Holm يخص اختبارات المقارنات الزوجية المقررة ولا يحوّل النتائج إلى دليل عام.
- لا تختبر هذه التجربة ربحية استراتيجية تداول. راجع `TECHNICAL_REPORT.md` و`VALIDATION_STATUS.md`.